In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import rasterio
import rioxarray
import boto3
import earthaccess
import requests
import geopandas as gpd
import pandas as pd
from pyproj import Transformer
from sentinel_tiles import UTC_to_solar, sentinel_tiles
import pystac_client
import lazycogs
import shapely
from functools import reduce
import obstore
import stac_geoparquet
import rustac
from tqdm.autonotebook import tqdm

earthaccess.login()

## Get MGRS grids for sample locations

In [ ]:
detections = xr.open_dataset("detections.nc")
transformer = Transformer.from_crs("EPSG:5071", "EPSG:4326", always_xy=True)

In [ ]:
bounds = [
    shapely.geometry.box(
        *transformer.transform_bounds(xmin, ymin, xmax, ymax)
    )
    for (xmin, ymin, xmax, ymax) in 
    zip(
     detections.xmin, detections.ymin, detections.xmax, detections.ymax   
    )
]

In [ ]:
mgrs_tile = np.array(list(map(lambda x: sentinel_tiles.tiles(x), bounds)))

In [ ]:
unique_tiles = reduce(set.union, mgrs_tile)

In [ ]:
len(unique_tiles)

In [ ]:
this_grid = next(iter(unique_tiles))
this_footprint = sentinel_tiles.bbox(this_grid)
this_footprint.buffer(-10000)

In [ ]:
print(this_grid)

## How fast can we read a grid?

In [ ]:
raw_items = await rustac.search(
    "https://cmr.earthdata.nasa.gov/stac/LPCLOUD",
    collections="ECO_L3T_MET_002",
    bbox=tuple(this_footprint.buffer(-25000).transform("EPSG:4326"))
)

print("Found", len(raw_items), "items")

For lazycogs to load things properly, asset keys need to be the same across granules. We only want assets on S3 as well.

In [ ]:
from copy import deepcopy

def sanitize_asset_keys(feature: dict):
    # Remake the asset dict
    new_asset_dict = {}
    for key in feature["assets"]:
        if key.startswith("s3"):
            new_key = key.split("_")[-1]
            new_asset_dict[new_key] = deepcopy(feature["assets"][key])

    # Copy all other properties over
    new_feature = deepcopy(feature)
    new_feature["assets"] = new_asset_dict

    return new_feature

sanitized_features = list(map(sanitize_asset_keys, raw_items))     

In [ ]:
sanitized_features[0]["assets"].keys()

In [ ]:
features_gpd = stac_geoparquet.to_geodataframe(sanitized_features)

In [ ]:
features_gpd.head()

In [ ]:
# Calculate solar datetime, only include items between 1 - 4 pm
tile_centroid_lon = shapely.centroid(features_gpd.geometry[0]).x
item_local_solar_time = features_gpd.start_datetime.apply(lambda x: UTC_to_solar(x, tile_centroid_lon))
features_gpd_time_filter = features_gpd[item_local_solar_time.dt.hour.isin([13, 14, 15, 16, 17])]

In [ ]:
features_gpd_time_filter.to_parquet("ta_items.parquet")

In [ ]:
dst_crs = this_footprint.crs
transformer = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
dst_bbox = transformer.transform_bounds(*bounds[0].bounds)
print(dst_bbox)

In [ ]:
from obstore.store import S3Store

creds = requests.get("https://data.lpdaac.earthdatacloud.nasa.gov/s3credentials").json()
s3_config = dict(
    aws_access_key_id=creds["accessKeyId"],
    aws_secret_access_key=creds["secretAccessKey"],
    aws_session_token=creds["sessionToken"]
)
store = S3Store(
    config=s3_config,
    bucket="lp-prod-protected"
)

In [ ]:
da = lazycogs.open(
    "ta_items.parquet",
    bbox=tuple(this_footprint.transform("EPSG:5071")),
    crs="EPSG:5071",
    resolution=70,
    store=store
)

In [ ]:
da

In [ ]:
# Find intersecting detections
detections_to_load = np.where(mgrs_tile == {this_grid,})[0]
detections_to_load

In [ ]:
slicers = [
    dict(
        x=slice(detections.xmin.data[idx], detections.xmax.data[idx]+300),
        y=slice(detections.ymin.data[idx], detections.ymax.data[idx]-300)
    )
    for idx in detections_to_load
]

In [ ]:
%%time
all_ts = [
    da.sel(**s).load().mean(dim=["x", "y"])
    for s in tqdm(slicers)
]

In [ ]:
(all_ts[0].sel(band="cloud") < 0.25).sum()

In [ ]:
all_ts[0].plot(x="time", row="band", sharey=False)

In [ ]:
albers_to_wgs = Transformer.from_crs("EPSG:5071", "EPSG:4326", always_xy=True)
point_lon = albers_to_wgs.transform(slicers[0]["x"].start, slicers[0]["y"].start)[1]
UTC_to_solar(all_ts[0].time.data, point_lon)